In [1]:
import Pkg; Pkg.add(["Ipopt", "SpecialFunctions", "DataFrames"])


   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


In [2]:
using Random, Distributions, JuMP, Ipopt, SpecialFunctions, DataFrames


# Parameters

This section sets the model's fundamentals and structural parameters, and fixes the indexing convention used throughout the rest of the notebook.

**Regions and markets.** There are $N$ regions and $J$ real sectors, plus a non-employment "sector 0" in every region, giving $M = J+1$ markets per region. Any array indexed over every possible market a household could occupy (labor `L`, value function `V`, migration shares `mu`, mobility costs `tau_mig`) is `N × M`, with **column 1 = non-employment** and **columns 2:M = real sectors 1..J**. Arrays that only pertain to production (`A_0`, `w_0`, `kappa_0`, `theta`, `eta`, `gamma`, `alpha`) are `N × J`, since sector 0 has no production, wage, or price — to compare a market-indexed array against a sector-indexed one, offset the sector index by $+1$ (sector `j`'s column in a production array corresponds to column `j+1` in a market array).

**Time-varying fundamentals**, $\Theta_t = (A_t, \kappa_t)$:
- $A_t^{nj}$ (coded as `A_0[n,j]`, its period-0 value) — productivity in region $n$, sector $j$
- $\kappa_t^{nj,ij}$ (coded as `kappa_0[j][n,i]`) — iceberg trade cost shipping sector-$j$ goods from region $i$ to region $n$

**Constant fundamentals**, $\bar\Theta = (\Upsilon, b)$:
- $\Upsilon = \{\tau^{nj,ik}\}$ (coded as `tau_mig`) — labor relocation (mobility) costs, in utility terms, from market $(n,j)$ to market $(i,k)$
- $b^n$ (coded as `b`) — value of home production in region $n$ (a non-employed household's consumption)

**Structural parameters:**
- $\beta \in [0,1)$ (`beta`) — discount factor
- $\theta^j$ (`theta`) — Fréchet trade elasticity in sector $j$
- $\nu$ (`nu`) — dispersion of the idiosyncratic migration taste shock ($1/\nu$ is the migration elasticity)
- $\alpha^j$ (`alpha`) — Cobb-Douglas consumption share on sector $j$, with $\sum_j \alpha^j = 1$

**State variable:** $L_t = \{L_t^{nj}\}$ (coded as `L_0` at $t=0$), the mass of households in each market at time $t$ — the only object carrying information from one period to the next.

In [ ]:
N = 2 # Number of regions
J = 3 # Number of real sectors (sector 0 / non-employment is handled separately -- see M below)
M = J + 1 # Number of markets per region: non-employment (market column 1) plus the J real
          # sectors (market columns 2:M). See the indexing convention above.
n_omega = 10000 # number of varieties used to discretize the omega in [0,1] continuum per region-sector


L_0  = ones(N,M) #labor force in economy at time 0, over all N regions x M markets (incl. non-employment)

Random.seed!(1) # fixed seed, for reproducibility

A_0 = rand(N,J) #region-sector productivity (real sectors only; undefined for non-employment)

B = ones(N,J) #arbitrary coefficient

w_0 = ones(N,J) # wage, real sectors only -- non-employment pays no wage; households there consume b_n instead

b = ones(N) # value of home production: consumption of a non-employed household in region n

kappa_0 = [ones(N,N) for _ in 1:J] # iceberg trade costs, per sector (kappa: trade cost)

theta = fill(4.0, J) # Frechet shape parameter per sector (governs dispersion of productivity draws
                      # across the continuum of varieties/regions, and the trade elasticity);
                      # scale is absorbed into A_0, so no separate T parameter is needed

eta = fill(2.0, N, J) # elasticity of substitution across varieties within sector j (CES aggregator)

gamma = ones(N, J) # labor/value-added share of production; = 1 since this simplified model has no materials

beta = 0.95 # household discount factor
# utility cost of moving from market (n,j) to market (i,k), j,k = 0,...,J (market columns 1:M,
# where column 1 is non-employment); 0 to stay in the same market, 1 otherwise (tau: migration cost)
tau_mig = [(n == i && j == k) ? 0.0 : 1.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]

alpha = rand(J) # Cobb-Douglas consumption shares: alpha[g] is the share of household income
alpha = alpha ./ sum(alpha) # spent on good g, common across all regions/sectors; normalized to sum to 1

nu = 1.0 # dispersion of the idiosyncratic (Frechet/type-I EV) taste shock over migration destinations
         # (larger nu = migration is less sensitive to utility differences, i.e. more friction)

T = 10 # number of periods to simulate the labor dynamics forward


## Baseline levels

Computes the reference point everything later in the notebook is defined relative to: the market-clearing wage `w_temp`, the trade shares `pi_temp` it implies (Definition 1), and (below) the migration shares of the model's own stationary equilibrium.

`solve_temporary_equilibrium` finds the wage that clears every market (eqs 5-7). There's no closed-form solution -- wages appear on both sides through trade shares -- so the code builds the wage as unknowns in a JuMP optimization model, writes eq. 7 as one constraint per market, and hands the whole system to the Ipopt solver with no real objective: it just searches for wage values that satisfy every constraint at once.

`stationary_V` (below) then computes each market's expected lifetime value by **iterating**: it starts from a guess, repeatedly recomputes the value from itself using the Bellman equation, and stops once the guess stops changing by more than a tiny tolerance. `migration_shares` is a single closed-form calculation (no loop) that turns that value function into a moving probability between every pair of markets.

In [ ]:
function trade_shares_and_expenditure(w::AbstractMatrix, L::AbstractMatrix)
    x = B .* w # unit cost in each region-sector: coefficient B times wage w
    trade_cost_term = [
        (x[i,j] * kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j]*gamma[i,j])
        for n in 1:N, j in 1:J, i in 1:N
    ] # for every (destination n, sector j, source i): multiply unit cost by the iceberg trade cost,
      # raise to -theta, multiply by productivity raised to theta*gamma -- the un-normalized "how
      # attractive is source i" term for eq 5
    pi = [trade_cost_term[n,j,i] / sum(trade_cost_term[n,j,:]) for n in 1:N, j in 1:J, i in 1:N]
      # eq 5: divide each source's term by the sum across all sources -- turns it into a SHARE
      # (the fraction of region n's sector-j spending that goes to source i)
    I = vec(sum(w .* L[:, 2:M], dims=2)) # total labor income per region: wage times employed labor,
      # summed across sectors (L[:,2:M] drops the non-employment column)
    X = I * alpha' # eq 6: multiplies each region's income by the consumption-share vector alpha,
      # giving an N x J matrix of region-sector expenditure
    return pi, X
end

function solve_temporary_equilibrium(L::AbstractMatrix; w_guess = ones(N,J))
    model = Model(Ipopt.Optimizer) # sets up an empty model that Ipopt (a nonlinear solver) will search
    set_silent(model)              # turns off Ipopt's solver log output

    @variable(model, w[n=1:N, j=1:J] >= 1e-6, start = w_guess[n,j])
      # declares w as the N x J unknowns the solver searches over, starting from w_guess

    @expression(model, tct[n=1:N, j=1:J, i=1:N],
        (B[i,j]*w[i,j]*kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j]*gamma[i,j]))
      # same formula as trade_cost_term above, written in terms of the solver's w so it re-evaluates
      # automatically as Ipopt tries different wage values
    @expression(model, pishare[n=1:N, j=1:J, i=1:N], tct[n,j,i] / sum(tct[n,j,m] for m in 1:N))
      # same normalization as pi above, in terms of w
    @expression(model, Inc[n=1:N], sum(w[n,k]*L[n,k+1] for k in 1:J)) # each region's labor income, in terms of w
    @expression(model, X[n=1:N, j=1:J], alpha[j]*Inc[n]) # each region-sector's expenditure, in terms of w

    @constraint(model, w[1,1] == 1.0) # fixes region 1 sector 1's wage to 1 (price numeraire)
    for n in 1:N, j in 1:J
        if !(n == 1 && j == 1)
            @constraint(model, w[n,j]*L[n,j+1] == sum(pishare[i,j,n]*X[i,j] for i in 1:N))
              # for every other market: requires the wage bill paid there to equal the total spending
              # other regions direct toward it -- this is the equation that "clears" the market
        end
    end

    @objective(model, Min, 0) # no real objective -- minimizing a constant turns this into pure
                               # constraint satisfaction: Ipopt just needs any w that meets every
                               # constraint above, not an optimal one
    optimize!(model) # runs Ipopt, searching over w until every constraint is satisfied

    w_star = value.(w) # reads the solved wage values out of the model
    pi_star, X_star = trade_shares_and_expenditure(w_star, L) # recomputes shares/expenditure at the
      # solved wage using the plain formula above
    return w_star, pi_star, X_star, termination_status(model) # also reports whether Ipopt actually converged
end


In [5]:
# Solve the temporary equilibrium at the initial labor distribution L_0 -- this is the
# baseline (w_temp, pi_temp) that the hat-algebra system below is defined relative to.

w_temp, pi_temp, X_temp, status_temp = solve_temporary_equilibrium(L_0)
println("Solver status: ", status_temp)



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

Solver status: LOCALLY_SOLVED


In [ ]:
function stationary_V(U_mkt::AbstractMatrix; tol=1e-12, maxiter=10_000)
    V = log.(U_mkt) # starting guess for the value function: log of flow utility at every market
    for _ in 1:maxiter
        V_next = [
            log(U_mkt[n,j]) + nu * log(sum(exp((beta*V[i,k] - tau_mig[n,j,i,k]) / nu) for i in 1:N, k in 1:M))
            for n in 1:N, j in 1:M
        ] # eq for V: for every market (n,j), takes log flow utility, plus nu times the log of a
          # sum over every possible destination (i,k) of exp((discounted continuation value minus the
          # moving cost to get there) / nu) -- recomputes V from the CURRENT V on the right-hand side
        if maximum(abs.(V_next .- V)) < tol
            V = V_next
            break # stops once the largest change across every market is below tol
        end
        V = V_next # otherwise, keeps the updated guess and loops again
    end
    return V
end

function migration_shares(V_next::AbstractMatrix)
    [
        exp((beta*V_next[i,k] - tau_mig[n,j,i,k]) / nu) /
        sum(exp((beta*V_next[m,h] - tau_mig[n,j,m,h]) / nu) for m in 1:N, h in 1:M)
        for n in 1:N, j in 1:M, i in 1:N, k in 1:M
    ] # eq 3: for every (origin (n,j), destination (i,k)) pair, exponentiates the discounted value at
      # the destination minus the cost of moving there, then divides by the sum of that same
      # quantity over every possible destination -- a single closed-form calculation, not a loop
end


# Dynamic Hat Algebra

Everything above is written in terms of *levels* (actual values of $A$, $\kappa$, $w$). This section rewrites the same two blocks -- the production side and the household's migration decision -- in terms of *proportional changes* relative to those levels instead, since a counterfactual is usually given as a change ("productivity up 20%") rather than a new absolute level. In the code, every such ratio is named with a `_dot` suffix (`w_dot`, `A_dot`, `L_dot`, `u_dot`, ...).

Mechanically, the two functions defined in this section (`solve_temp_eq_hat` and `stationary_u_hat`/`migration_shares_next`) work exactly like `solve_temporary_equilibrium` and `stationary_V`/`migration_shares` above -- same JuMP/Ipopt solve, same fixed-point iteration -- just applied to the ratio versions of the equations. They're the building blocks that `solve_transition_path_hat` (last section of this notebook) calls once per period to trace out a full transition path.

## Temporary equilibrium in time differences

Ratio-form version of the production block above: given a baseline allocation $(w_t, \pi_t, L_t)$ and a change in labor supply $\dot L_{t+1}=L_{t+1}/L_t$ and fundamentals $\dot A_{t+1}, \dot\kappa_{t+1}$, solves for the wage change $\dot w_{t+1}=w_{t+1}/w_t$.

$$\dot x_{t+1}^{nj} = \dot w_{t+1}^{nj} \tag{8}$$
$$\dot P_{t+1}^{nj} = \left(\sum_{i=1}^N \pi_t^{nj,ij}\left(\dot w_{t+1}^{ij}\dot\kappa_{t+1}^{nj,ij}\right)^{-\theta^j}\left(\dot A_{t+1}^{ij}\right)^{\theta^j}\right)^{-1/\theta^j} \tag{9}$$
$$\pi_{t+1}^{nj,ij} = \pi_t^{nj,ij}\left(\frac{\dot w_{t+1}^{ij}\dot\kappa_{t+1}^{nj,ij}}{\dot P_{t+1}^{nj}}\right)^{-\theta^j}\left(\dot A_{t+1}^{ij}\right)^{\theta^j} \tag{10}$$
$$X_{t+1}^{nj} = \alpha^j\sum_{k=1}^J \dot w_{t+1}^{nk}\dot L_{t+1}^{nk}\, w_t^{nk}L_t^{nk} \tag{11}$$
$$\dot w_{t+1}^{nj}\dot L_{t+1}^{nj}\, w_t^{nj}L_t^{nj} = \sum_{i=1}^N \pi_{t+1}^{ij,nj} X_{t+1}^{ij} \tag{12}$$

`solve_temp_eq_hat` (below) is coded exactly like `solve_temporary_equilibrium` above: it declares `w_dot` as unknowns in a JuMP model, writes eq. (9)-(11) as plain formulas (`@expression`) in terms of `w_dot`, writes eq. (12) as one constraint per market, and calls Ipopt to find a `w_dot` that satisfies all of them -- again a pure constraint-satisfaction solve, not an optimization. Everything else (`P_dot`, `pi_next`, `X_next`) is then read straight off the solved model, since eqs (9)-(11) already express them directly in terms of `w_dot`.

In [ ]:
function solve_temp_eq_hat(pi_t::Array{Float64,3}, w_t::AbstractMatrix, L_t::AbstractMatrix,
                            L_dot::AbstractMatrix, A_dot::AbstractMatrix, kappa_dot;
                            w_dot_guess = ones(N,J))
    model = Model(Ipopt.Optimizer) # sets up an empty model for Ipopt to search
    set_silent(model)              # turns off Ipopt's solver log output
    @variable(model, w_dot[n=1:N, j=1:J] >= 1e-6, start = w_dot_guess[n,j])
      # declares w_dot as the N x J unknowns the solver searches over

    @expression(model, P_dot[n=1:N, j=1:J],
        (sum(pi_t[n,j,i] * (w_dot[i,j]*kappa_dot[j][n,i])^(-theta[j]) * A_dot[i,j]^theta[j] for i in 1:N))^(-1/theta[j]))
      # eq 9: for each market, sums over every source i the baseline trade share times (wage change
      # times trade-cost change) raised to -theta times productivity change raised to theta, then
      # raises the whole sum to -1/theta -- the price-index change
    @expression(model, pi_next[n=1:N, j=1:J, i=1:N],
        pi_t[n,j,i] * ((w_dot[i,j]*kappa_dot[j][n,i]) / P_dot[n,j])^(-theta[j]) * A_dot[i,j]^theta[j])
      # eq 10: multiplies the baseline trade share by (relative cost change / price index change)
      # raised to -theta and by productivity change raised to theta -- gives the NEW trade share
      # LEVEL directly (not a further ratio), since pi_t is already multiplied in
    @expression(model, X_next[n=1:N, j=1:J],
        alpha[j] * sum(w_dot[n,k]*L_dot[n,k+1]*w_t[n,k]*L_t[n,k+1] for k in 1:J))
      # eq 11: multiplies the consumption share alpha by the sum, over every sector k, of the NEW
      # wage-bill level in that sector (wage change x labor change x last period's wage x last
      # period's labor) -- also a LEVEL, built entirely from period-t values and the given changes

    @constraint(model, w_dot[1,1] == 1.0) # fixes region 1 sector 1's wage change to 1 (numeraire)
    for n in 1:N, j in 1:J
        if !(n == 1 && j == 1)
            @constraint(model, w_dot[n,j]*L_dot[n,j+1]*w_t[n,j]*L_t[n,j+1] == sum(pi_next[i,j,n]*X_next[i,j] for i in 1:N))
              # eq 12: for every other market, requires the new wage-bill level there to equal the
              # total new expenditure other regions direct toward it -- both sides are LEVELS, so
              # pi_next/X_next can be used directly with no further multiplication by pi_t
        end
    end

    @objective(model, Min, 0) # no real objective -- pure constraint satisfaction, same as above
    optimize!(model) # runs Ipopt, searching over w_dot until every constraint is satisfied

    return value.(w_dot), value.(P_dot), value.(pi_next), value.(X_next), termination_status(model)
      # reads the solved w_dot, P_dot, pi_next, X_next out of the model, plus whether Ipopt converged
end


## Sequential equilibrium in time differences (household block)

Ratio-form version of the household migration decision above: $\mu_t^{nj,ik}$ is the share of households in market $(n,j)$ who move to market $(i,k)$ next period, and $u_t^{nj}\equiv\exp(V_t^{nj})$.

$$\mu_{t+1}^{nj,ik} = \frac{\mu_t^{nj,ik}\left(\dot u_{t+2}^{ik}\right)^{\beta/\nu}}{\sum_{m=1}^N\sum_{h=0}^J \mu_t^{nj,mh}\left(\dot u_{t+2}^{mh}\right)^{\beta/\nu}} \tag{13}$$
$$\dot u_{t+1}^{nj} = \dot\omega^{nj}(\dot L_{t+1},\dot\Theta_{t+1})\left(\sum_{i=1}^N\sum_{k=0}^J \mu_t^{nj,ik}\left(\dot u_{t+2}^{ik}\right)^{\beta/\nu}\right)^{\nu} \tag{14}$$
$$L_{t+1}^{nj} = \sum_{i=1}^N\sum_{k=0}^J \mu_t^{ik,nj}L_t^{ik} \tag{15}$$

$\dot\omega_{t+1}^{nj}=\dot w_{t+1}^{nj}/\dot P_{t+1}^n$ (real wage change) is what connects this block to the production block above -- `solve_temp_eq_hat`'s output feeds eq. 14 directly. Non-employment pays no wage, so $\dot\omega_{t+1}^{n0}=1$ always.

`stationary_u_hat` (below) is coded exactly like `stationary_V` above: it starts from a guess (`u_dot = 1` everywhere), repeatedly recomputes eq. 14's right-hand side from the current guess, and stops once the guess stops changing by more than `tol` -- just multiplying instead of adding at each step. `migration_shares_next` then applies eq. 13 directly to an already-solved `u_dot`: a single closed-form calculation, not a loop.

In [ ]:
function stationary_u_hat(omega_dot::AbstractMatrix, mu_baseline::Array{Float64,4}; tol=1e-12, maxiter=10_000)
    u_dot = ones(N, M) # starting guess: assume no change at every market
    for _ in 1:maxiter
        u_dot_next = [
            omega_dot[n,j] * (sum(mu_baseline[n,j,i,k] * u_dot[i,k]^(beta/nu) for i in 1:N, k in 1:M))^nu
            for n in 1:N, j in 1:M
        ] # eq 14: for every market, multiplies the real wage change by a migration-share-weighted
          # sum over every destination of (that destination's current u_dot guess, raised to
          # beta/nu), then raises the whole sum to nu -- recomputes u_dot from the CURRENT u_dot
        converged = maximum(abs.(u_dot_next .- u_dot)) < tol # true once the largest change across
                                                               # every market is below tol
        u_dot = u_dot_next # keeps the updated guess either way
        converged && break # stops the loop if converged, otherwise tries again
    end
    return u_dot
end

function migration_shares_next(u_dot::AbstractMatrix, mu_baseline::Array{Float64,4})
    [
        mu_baseline[n,j,i,k] * u_dot[i,k]^(beta/nu) /
        sum(mu_baseline[n,j,m,h] * u_dot[m,h]^(beta/nu) for m in 1:N, h in 1:M)
        for n in 1:N, j in 1:M, i in 1:N, k in 1:M
    ] # eq 13: for every (origin, destination) pair, multiplies last period's migration share by
      # (this destination's u_dot raised to beta/nu), then divides by that same product summed over
      # every possible destination -- a single closed-form calculation using an already-solved u_dot
end

function flow_utility_mkt_at(w::AbstractMatrix, A::AbstractMatrix, kappa)
    x = B .* w # unit cost in each region-sector, at the given wage w
    trade_cost_term = [
        (x[i,j] * kappa[j][n,i])^(-theta[j]) * A[i,j]^(theta[j]*gamma[i,j])
        for n in 1:N, j in 1:J, i in 1:N
    ] # same trade-cost-term formula as trade_shares_and_expenditure above, but using the w/A/kappa
      # passed in as arguments instead of the module-level w/A_0/kappa_0
    Gamma = [SpecialFunctions.gamma((theta[j] + 1 - eta[n,j]) / theta[j])^(1 / (1 - eta[n,j])) for n in 1:N, j in 1:J]
      # a CES/Frechet aggregation constant, computed once per region-sector from theta and eta
    P = [Gamma[n,j] * sum(trade_cost_term[n,j,:])^(-1/theta[j]) for n in 1:N, j in 1:J]
      # multiplies that constant by (the trade-cost terms summed across every source, raised to
      # -1/theta) -- the sector-level price index in each region
    P_hat_region = [prod((P[n,g] / alpha[g])^alpha[g] for g in 1:J) for n in 1:N]
      # region-level price index: the product, across every sector g, of (that sector's price
      # divided by its consumption share) raised to that same consumption share
    C = [w[n,j] / P_hat_region[n] for n in 1:N, j in 1:J] # real wage: nominal wage divided by the
                                                            # region-level price index
    return hcat(b, C) # prepends the home-production value b as the non-employment column, so the
                       # result lines up with the N x M market indexing used everywhere else
end

# Solving for the Transition Path

Given a baseline allocation $(L_0,\pi_0,w_0)$, last period's migration shares $\mu_{-1}$, and a path of fundamental changes $\dot\Theta_t=(\dot A_t,\dot\kappa_t)$ for $t=1,\dots,T$ (no further change assumed after $T$), `solve_transition_path_hat` (below) computes the whole transition path $\{L_t,\mu_t,w_t\}$ by chaining the two blocks above together period by period.

It reuses eqs 8-15 exactly as defined above -- no new equations here, just a loop that calls `solve_temp_eq_hat` and applies eqs 13-15 repeatedly.

**Why there's an outer loop.** The migration-share path $\mu_t$ needs $\dot u_{t+1}$ to be simulated forward (eq. 13), but $\dot u_t$ needs the full $\mu$ path to be solved backward (eq. 14) -- neither can be computed first. The code handles this by *guessing* a path for $\dot u_t$, running one complete forward-then-backward pass to get an updated path, and repeating until the guess and the result agree. One pass through the outer `for outer in 1:max_outer` loop does:

1. **Forward, using the current guess:** simulate $\mu_0,...,\mu_T$ (eq. 13) from $\mu_{-1}$, then simulate $L_1,...,L_{T+1}$ (eq. 15) from $L_0$ using that $\mu$ path. Both are direct, non-iterative loops over $t=0,\dots,T$ -- each period computed straight from the previous one.
2. **Forward again, solving the production side:** loop over $t=0,\dots,T$ a second time, calling `solve_temp_eq_hat` once per period (a fresh JuMP/Ipopt solve each time) against that period's running wage/trade-share levels, then updating those levels before moving to $t+1$. Levels (not just growth rates) have to be carried through this loop because eq. 11's $X_{t+1}$ needs the actual wage-bill level $w_t L_t$, which compounds period over period.
3. **Backward:** loop over $t=T-1,\dots,0$ -- counting *down* -- applying eq. 14 starting from the fixed terminal condition $\dot u_{T+1}=1$, since $\dot u_{t+1}$ depends on the *next* period's $\dot u_{t+2}$. This produces an updated guess for the whole $\dot u$ path.
4. **Compare and update:** measure the largest change anywhere between the updated path and the guess used to run steps 1-2, then set the next guess to a *weighted average* of the two rather than replacing it outright -- see the `damp` comment in the code below for why. Repeat from step 1 until that largest change is below `tol`, or `max_outer` passes are used up.

In [ ]:
# Indexing note (Julia arrays are 1-indexed, but the model's time index starts at 0 or -1 for
# several objects): L_path[s] holds L_{s-1}, mu_path[s] holds mu_{s-1}, u_dot_guess[s] holds
# u_dot_s directly, and A_dot_path[s]/kappa_dot_path[s] hold the shock going INTO period s.

function solve_transition_path_hat(w_0::AbstractMatrix, L_0::AbstractMatrix, pi_0::Array{Float64,3},
                                    mu_minus1::Array{Float64,4}, A_dot_path::Vector, kappa_dot_path::Vector;
                                    T::Int, max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5)

    u_dot_guess = [ones(N,M) for _ in 1:T] # starting guess: assume no change (u_dot=1) every period
    u_dot_terminal = ones(N,M)             # fixed assumption: nothing changes after period T

    local mu_path, L_path, omega_dot_path, w_path, pi_final, converged, outer_used

    for outer in 1:max_outer   # repeats the forward+backward pass below until the guess stops moving

        # --- forward pass 1: build the migration-share path using the CURRENT guess of u_dot ---
        mu_path = Vector{Array{Float64,4}}(undef, T+1) # allocates space for mu_0,...,mu_T
        mu_prev = mu_minus1 # starts the walk-forward from last period's realized shares
        for t in 0:T
            u_dot_tp1 = (t+1 <= T) ? u_dot_guess[t+1] : u_dot_terminal # looks up next period's
                                                                        # guessed u_dot (or the
                                                                        # fixed terminal value at t=T)
            mu_path[t+1] = migration_shares_next(u_dot_tp1, mu_prev) # eq 13: computes this period's
                                                                      # migration shares from last
                                                                      # period's shares and u_dot
            mu_prev = mu_path[t+1] # this period's result becomes "previous" for the next loop step
        end

        # --- forward pass 2: build the labor-distribution path from the mu path just computed ---
        L_path = Vector{Matrix{Float64}}(undef, T+2) # allocates space for L_0,...,L_{T+1}
        L_path[1] = L_0
        for t in 0:T
            L_path[t+2] = [
                sum(mu_path[t+1][i,k,n,j] * L_path[t+1][i,k] for i in 1:N, k in 1:M)
                for n in 1:N, j in 1:M
            ] # eq 15: for every destination market (n,j), sums over every origin market (i,k) the
              # number of people there times the share of them that moves to (n,j)
        end

        # --- forward pass 3: solve the production side one period at a time, carrying levels forward ---
        omega_dot_path = Vector{Matrix{Float64}}(undef, T+1)
        w_path = Vector{Matrix{Float64}}(undef, T+2)
        w_path[1] = w_0        # starts the wage-LEVEL path at the baseline
        pi_current = pi_0      # starts the trade-share LEVEL at the baseline
        for t in 0:T
            L_dot_step = [
                L_path[t+1][n,j] == 0 ? 1.0 : L_path[t+2][n,j] / L_path[t+1][n,j]
                for n in 1:N, j in 1:M
            ] # this period's labor growth rate, market by market; uses 1.0 instead of dividing by
              # zero when a market is empty (it contributes nothing to the constraint either way)
            A_dot_step = (t+1 <= T) ? A_dot_path[t+1] : ones(N,J)              # this period's productivity shock
            kappa_dot_step = (t+1 <= T) ? kappa_dot_path[t+1] : [ones(N,N) for _ in 1:J] # this period's trade-cost shock

            w_dot, P_dot, pi_next, X_next, status = solve_temp_eq_hat(
                pi_current, w_path[t+1], L_path[t+1], L_dot_step, A_dot_step, kappa_dot_step)
              # solves for this period's wage change -- the same JuMP/Ipopt solve as the
              # single-period function above, called once per period inside this loop

            P_hat_dot = [prod(P_dot[n,k]^alpha[k] for k in 1:J) for n in 1:N]
              # region-level price index change: product across sectors of each sector's price
              # change raised to its consumption share
            omega_dot_path[t+1] = hcat(ones(N), w_dot ./ P_hat_dot)
              # real wage change: nominal wage change divided by the region price index change
              # (a column of 1's is prepended for non-employment, which has no wage to deflate)
            w_path[t+2] = w_dot .* w_path[t+1] # updates the wage LEVEL: multiplies last period's
                                                # level by this period's solved change
            pi_current = pi_next # carries the updated trade-share level forward to the next period
        end
        pi_final = pi_current

        # --- backward pass: solves for u_dot from the LAST period back to the first ---
        u_dot_new = Vector{Matrix{Float64}}(undef, T)
        u_dot_next = u_dot_terminal # starts from the fixed terminal condition, u_dot_{T+1}=1
        for t in (T-1):-1:0   # counts DOWN from T-1 to 0 -- backward induction
            u_dot_new[t+1] = [
                omega_dot_path[t+1][n,j] * (sum(mu_path[t+1][n,j,i,k] * u_dot_next[i,k]^(beta/nu) for i in 1:N, k in 1:M))^nu
                for n in 1:N, j in 1:M
            ] # eq 14: multiplies the real wage change by a migration-share-weighted sum over every
              # destination of (next period's u_dot raised to beta/nu), then raises that sum to nu
            u_dot_next = u_dot_new[t+1] # this period's result becomes "next period" for the
                                         # following, earlier loop step
        end

        # --- compares the freshly solved path to the guess, then updates the guess ---
        diff = maximum(maximum(abs.(u_dot_new[t] .- u_dot_guess[t])) for t in 1:T)
          # largest change anywhere in the path, comparing the new solve to the guess that produced it
        u_dot_guess = [damp .* u_dot_new[t] .+ (1-damp) .* u_dot_guess[t] for t in 1:T]
          # sets the next guess to a WEIGHTED AVERAGE of the new result and the old guess, instead
          # of replacing it outright: moves only `damp` of the way there each round (default 0.5,
          # i.e. halfway). This forward/backward-coupled loop can overshoot and oscillate if it
          # jumps straight to the new guess every time (damp=1); blending in only a fraction each
          # round trades slower convergence for stability
        converged = diff < tol # true once the guess and the freshly solved path agree closely enough
        outer_used = outer
        if converged
            println("solve_transition_path_hat converged after $outer outer iterations (max u_dot change = $diff)")
            break # stops the outer loop early once converged
        end
        if outer == max_outer
            println("solve_transition_path_hat: reached max_outer=$max_outer without converging (max u_dot change = $diff)")
            # ran out of allowed outer iterations without diff dropping below tol
        end
    end

    return (; L_path, mu_path, w_path, pi_final, u_dot_path = u_dot_guess, omega_dot_path, converged, outer_used)
      # packages up the full solved path, plus whether/how fast the outer loop converged
end
